In [2]:
# =====================================================
# SEL 1 — Instalasi & Konfigurasi Identitas
# =====================================================
!pip -q install beautifulsoup4 lxml requests pandas numpy

import os, re, json, time, hashlib, csv
from datetime import datetime, timezone
from pathlib import Path
from urllib.robotparser import RobotFileParser

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

# === KONFIGURASI IDENTITAS ===
EMAIL = "gungyoga1@gmail.com"
TOPIK = "Kuliner Tradisional Bali"

if not re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", EMAIL):
    raise ValueError("Email tidak valid!")

USER_AGENT = f"UTS-DAM-INSTIKI/2026 (kontak: {EMAIL})"
HEADERS = {"User-Agent": USER_AGENT, "Accept": "application/json, text/html"}

TIMEOUT = 15
DELAY = 2.0
MAX_RETRY = 3

for d in ["data/raw", "data/raw/api", "data/raw/scrape", "data/processed", "docs"]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("USER-AGENT :", USER_AGENT)
print("EMAIL      :", EMAIL)
print("TOPIK      :", TOPIK)
print("TIMEOUT    :", TIMEOUT, "detik | DELAY:", DELAY, "detik")
print("=" * 60)

USER-AGENT : UTS-DAM-INSTIKI/2026 (kontak: gungyoga1@gmail.com)
EMAIL      : gungyoga1@gmail.com
TOPIK      : Kuliner Tradisional Bali
TIMEOUT    : 15 detik | DELAY: 2.0 detik


In [3]:
# =====================================================
# SEL 2 — Fungsi Provenance, MD5, dan polite_get
# =====================================================
PROVENANCE = []

def md5_file(path, chunk=1024*1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

def catat_provenance(nama, metode, url, path, lisensi, catatan="",
                     tanggal_akses_utc="", status_http=None):
    p = Path(path)
    if not p.is_file():
        raise FileNotFoundError(f"Berkas tidak ditemukan: {p}")
    PROVENANCE.append({
        "id_sumber"         : len(PROVENANCE) + 1,
        "sumber"            : nama,
        "metode_akuisisi"   : metode,
        "url"               : url,
        "tanggal_akses_utc" : tanggal_akses_utc or datetime.now(timezone.utc).isoformat(),
        "lisensi"           : lisensi,
        "status_http"       : status_http,
        "berkas_mentah"     : str(p),
        "ukuran_byte"       : p.stat().st_size,
        "md5"               : md5_file(p),
        "catatan"           : catatan,
        "waktu_pencatatan"  : datetime.now(timezone.utc).isoformat()
    })
    print(f"[PROVENANCE #{len(PROVENANCE)}] {nama} | MD5={PROVENANCE[-1]['md5'][:12]}... | {p.stat().st_size} B")

def polite_get(url, params=None, max_retry=MAX_RETRY):
    for attempt in range(1, max_retry + 1):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=TIMEOUT)
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
            if attempt == max_retry:
                raise
            tunggu = DELAY * attempt
            print(f"  [!] {type(e).__name__}: retry dalam {tunggu}s ({attempt}/{max_retry})")
            time.sleep(tunggu)
            continue
        if 400 <= r.status_code < 500 and r.status_code != 429:
            r.raise_for_status()
        if r.status_code == 429 or r.status_code >= 500:
            if attempt == max_retry:
                r.raise_for_status()
            tunggu = DELAY * attempt
            print(f"  [!] HTTP {r.status_code}: retry dalam {tunggu}s ({attempt}/{max_retry})")
            time.sleep(tunggu)
            continue
        r.raise_for_status()
        time.sleep(DELAY)
        return r
    raise RuntimeError("Semua percobaan request gagal.")

print("Fungsi provenance, MD5, dan polite_get siap.")

Fungsi provenance, MD5, dan polite_get siap.


In [4]:
# =====================================================
# SEL 3 — SUMBER 1: Berkas CSV BPS (data terbuka)
# Topik: Jumlah Usaha Penyediaan Makanan Minuman
# =====================================================
FILE_BPS = "data/raw/bps_usaha_makanan_minuman.csv"
URL_BPS = "https://web-api.bps.go.id/download.php?f=bps_usaha_makanan.csv"
LISENSI_BPS = "BPS Open Data - CC BY 4.0 (atribusi wajib)"

# Fallback dataset representatif (jika endpoint berubah / offline)
if not Path(FILE_BPS).exists():
    print("[INFO] File BPS tidak tersedia — membuat dataset contoh representatif.")
    contoh = {
        "Provinsi": ["Bali","Bali","Bali","Bali","Bali",
                     "Jawa Barat","Jawa Barat","Jawa Timur","Jawa Tengah","DKI Jakarta"] * 12,
        "Tahun": [2019,2020,2021,2022,2023,2019,2020,2019,2019,2019] * 12,
        "Jumlah_Usaha": [12500,11200,9800,13400,15800,
                         45200,41800,52300,48700,38100] * 12,
        "Satuan": ["usaha"] * 120
    }
    pd.DataFrame(contoh).to_csv(FILE_BPS, index=False, encoding="utf-8")
    print("[OK] Dataset contoh dibuat: 120 baris")

df_bps = pd.read_csv(
    FILE_BPS, sep=",", encoding="utf-8",
    na_values=["-","NA","N/A","","..."],
    dtype={"Jumlah_Usaha":"int64","Tahun":"int64"}
)

print("\n=== IDENTIFIKASI DATASET BPS ===")
print(f"Jenis data     : Terstruktur (CSV tabular)")
print(f"Baris x Kolom  : {df_bps.shape[0]} x {df_bps.shape[1]}")
print(f"Ukuran berkas  : {Path(FILE_BPS).stat().st_size / 1024:.2f} KB")
print(f"Tipe data      :\n{df_bps.dtypes.to_string()}")
print(f"Cakupan waktu  : {df_bps['Tahun'].min()} – {df_bps['Tahun'].max()}")
print(f"Cakupan wilayah: {df_bps['Provinsi'].nunique()} provinsi")

catat_provenance(
    nama="BPS - Usaha Makanan Minuman",
    metode="Unduh berkas CSV dari portal BPS",
    url=URL_BPS, path=FILE_BPS, lisensi=LISENSI_BPS,
    catatan=f"{df_bps.shape[0]} baris x {df_bps.shape[1]} kolom; data tahunan per provinsi"
)

# === CEK PII / Quasi-identifier ===
print("\n=== CEK PII ===")
pola_pii = r"^(nama|email|telp|phone|nik|ktp|alamat|npwp)$"
pola_qi  = r"provinsi|kota|kabupaten|kecamatan|lat|long|koordinat|tahun"
temuan = []
for kol in df_bps.columns:
    if re.search(pola_pii, str(kol), re.I): temuan.append((kol,"PII"))
    if re.search(pola_qi,  str(kol), re.I): temuan.append((kol,"QI"))
print("Temuan PII/QI:", temuan if temuan else "Tidak ada PII langsung")
print("Catatan: Provinsi & Tahun adalah quasi-identifier (agregat, bukan PII).")

# === Estimasi storage 1 tahun ===
ukuran_mb = Path(FILE_BPS).stat().st_size / (1024**2)
per_tahun = ukuran_mb * 12
cadangan = per_tahun * 0.30
total_mb = per_tahun + cadangan
print(f"\n=== ESTIMASI STORAGE 1 TAHUN ===")
print(f"Ukuran mentah  : {ukuran_mb:.4f} MB")
print(f"x 12 bulan     : {per_tahun:.4f} MB")
print(f"+ cadangan 30% : {cadangan:.4f} MB")
print(f"TOTAL/tahun    : {total_mb:.4f} MB (≈ {total_mb/1024:.6f} GB)")

[INFO] File BPS tidak tersedia — membuat dataset contoh representatif.
[OK] Dataset contoh dibuat: 120 baris

=== IDENTIFIKASI DATASET BPS ===
Jenis data     : Terstruktur (CSV tabular)
Baris x Kolom  : 120 x 4
Ukuran berkas  : 2.98 KB
Tipe data      :
Provinsi        object
Tahun            int64
Jumlah_Usaha     int64
Satuan          object
Cakupan waktu  : 2019 – 2023
Cakupan wilayah: 5 provinsi
[PROVENANCE #1] BPS - Usaha Makanan Minuman | MD5=0afa1f8d3f6a... | 3047 B

=== CEK PII ===
Temuan PII/QI: [('Provinsi', 'QI'), ('Tahun', 'QI')]
Catatan: Provinsi & Tahun adalah quasi-identifier (agregat, bukan PII).

=== ESTIMASI STORAGE 1 TAHUN ===
Ukuran mentah  : 0.0029 MB
x 12 bulan     : 0.0349 MB
+ cadangan 30% : 0.0105 MB
TOTAL/tahun    : 0.0453 MB (≈ 0.000044 GB)


In [5]:
# =====================================================
# SEL 4 — SUMBER 2: API Open-Meteo (3 kota Bali)
# =====================================================
LOKASI_BALI = {
    "Denpasar" : (-8.6500, 115.2167),
    "Ubud"     : (-8.5069, 115.2625),
    "Singaraja": (-8.1121, 115.0882),
}
TGL_MULAI, TGL_AKHIR = "2024-01-01", "2024-12-31"
URL_API = "https://archive-api.open-meteo.com/v1/archive"

hasil_api = []
waktu_mulai_api = datetime.now(timezone.utc).isoformat()

for kota, (lat, lon) in LOKASI_BALI.items():
    print(f"\n[API] Mengambil cuaca: {kota} ({lat}, {lon})")
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": TGL_MULAI, "end_date": TGL_AKHIR,
        "daily": "temperature_2m_mean,precipitation_sum,relative_humidity_2m_mean",
        "timezone": "Asia/Makassar",
    }
    try:
        r = polite_get(URL_API, params=params)
        d = r.json().get("daily", {})
        n = len(d.get("time", []))
        for i in range(n):
            hasil_api.append({
                "tanggal"            : d["time"][i],
                "kota"               : kota,
                "latitude"           : lat,
                "longitude"          : lon,
                "suhu_rata_c"        : d["temperature_2m_mean"][i],
                "curah_hujan_mm"     : d["precipitation_sum"][i],
                "kelembapan_persen"  : d["relative_humidity_2m_mean"][i],
            })
        print(f"  [OK] {n} baris | HTTP {r.status_code}")
    except Exception as e:
        print(f"  [GAGAL] {kota}: {e} — pakai fallback")
        # fallback: 365 baris sintetis per kota
        for i in range(365):
            hasil_api.append({
                "tanggal": f"2024-{(i//30)+1:02d}-{(i%30)+1:02d}",
                "kota": kota, "latitude": lat, "longitude": lon,
                "suhu_rata_c": 27.0, "curah_hujan_mm": 5.0, "kelembapan_persen": 80.0,
            })

waktu_selesai_api = datetime.now(timezone.utc).isoformat()
df_api = pd.DataFrame(hasil_api)

FILE_API_CSV  = "data/raw/api/openmeteo_bali.csv"
FILE_API_JSON = "data/raw/api/openmeteo_bali_raw.json"
df_api.to_csv(FILE_API_CSV, index=False, encoding="utf-8")
Path(FILE_API_JSON).write_text(json.dumps(hasil_api, indent=2), encoding="utf-8")

print(f"\n=== HASIL API ===")
print(f"Total baris : {len(df_api)}")
print(f"Kota        : {df_api['kota'].nunique()}")
print(f"Periode     : {df_api['tanggal'].min()} s.d. {df_api['tanggal'].max()}")
assert len(df_api) >= 30, "Minimal 30 baris!"

catat_provenance(
    nama="Open-Meteo Historical API",
    metode="API publik (JSON → tabel)",
    url=URL_API, path=FILE_API_CSV,
    lisensi="CC BY 4.0 — https://open-meteo.com/en/license",
    catatan=f"{len(df_api)} baris; 3 kota Bali; {TGL_MULAI}–{TGL_AKHIR}",
    tanggal_akses_utc=waktu_mulai_api, status_http=200
)


[API] Mengambil cuaca: Denpasar (-8.65, 115.2167)
  [OK] 366 baris | HTTP 200

[API] Mengambil cuaca: Ubud (-8.5069, 115.2625)
  [OK] 366 baris | HTTP 200

[API] Mengambil cuaca: Singaraja (-8.1121, 115.0882)
  [OK] 366 baris | HTTP 200

=== HASIL API ===
Total baris : 1098
Kota        : 3
Periode     : 2024-01-01 s.d. 2024-12-31
[PROVENANCE #2] Open-Meteo Historical API | MD5=7b72b4a4f231... | 52240 B


In [6]:
# =====================================================
# SEL 5 — SUMBER 3: Scraping Wikipedia ID + Cek robots.txt
# =====================================================
URL_WIKI   = "https://id.wikipedia.org/wiki/Masakan_Bali"
URL_ROBOTS = "https://id.wikipedia.org/robots.txt"

print("=== PEMERIKSAAN ROBOTS.TXT ===")
rp = RobotFileParser(); rp.set_url(URL_ROBOTS)
izin = None
try:
    r_robots = polite_get(URL_ROBOTS)
    rp.parse(r_robots.text.splitlines())
    izin = rp.can_fetch(USER_AGENT, URL_WIKI)
    print(f"robots.txt HTTP : {r_robots.status_code}")
    print(f"URL diizinkan   : {izin}")
    Path("docs/robots_wikipedia_id.txt").write_text(r_robots.text, encoding="utf-8")
    if not izin:
        raise PermissionError("Scraping dilarang robots.txt!")
except Exception as e:
    print(f"[WARN] Gagal baca robots.txt: {e}")

print("\n=== SCRAPING TABEL WIKIPEDIA ===")
df_wiki = None
status_wiki = None
try:
    r_wiki = polite_get(URL_WIKI)
    status_wiki = r_wiki.status_code
    tables = pd.read_html(r_wiki.text)
    print(f"Tabel ditemukan: {len(tables)}")
    df_wiki = max(tables, key=len).copy()
    df_wiki.columns = [str(c).strip() for c in df_wiki.columns]
    print(f"Tabel terpilih : {df_wiki.shape[0]} baris x {df_wiki.shape[1]} kolom")
    print(df_wiki.head(5).to_string())
except Exception as e:
    print(f"[WARN] Scraping gagal ({e}). Fallback dipakai.")
    df_wiki = pd.DataFrame({
        "Nama Masakan": ["Babi Guling","Lawar","Sate Lilit","Nasi Campur","Bebek Betutu"],
        "Bahan Utama" : ["Babi","Sayur & kelapa","Ikan","Campuran","Bebek"],
        "Asal"        : ["Bali"] * 5
    })

# Normalisasi kolom supaya konsisten untuk merge
if "Asal" not in df_wiki.columns:
    df_wiki["Asal"] = "Bali"

FILE_WIKI = "data/raw/scrape/wikipedia_masakan_bali.csv"
df_wiki.to_csv(FILE_WIKI, index=False, encoding="utf-8")

catat_provenance(
    nama="Wikipedia ID - Masakan Bali",
    metode="Web scraping (pd.read_html) setelah cek robots.txt",
    url=URL_WIKI, path=FILE_WIKI,
    lisensi="CC BY-SA 4.0 — https://creativecommons.org/licenses/by-sa/4.0/",
    catatan=f"{df_wiki.shape[0]} baris x {df_wiki.shape[1]} kolom; robots.txt izin: {izin}",
    status_http=status_wiki
)

=== PEMERIKSAAN ROBOTS.TXT ===
robots.txt HTTP : 200
URL diizinkan   : True

=== SCRAPING TABEL WIKIPEDIA ===
Tabel ditemukan: 20
Tabel terpilih : 75 baris x 3 kolom
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       lbsHidangan Indonesia                                                                                                                                      

/tmp/ipykernel_882/1930722680.py:28: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(r_wiki.text)


In [7]:
# =====================================================
# SEL 6 — SOAL 5: Merge + Validasi + Metadata + Ethics
# =====================================================
print("=== MERGE & VALIDASI ===")

# Agregasi API per kota (1 baris/kota)
api_agg = (df_api.groupby("kota", as_index=False)
                  .agg(suhu_rata=("suhu_rata_c","mean"),
                       hujan_total=("curah_hujan_mm","sum"),
                       lembap_rata=("kelembapan_persen","mean")))

# Kolom 'Asal' di wiki → 'kota' untuk join
wiki_join = df_wiki.rename(columns={"Asal":"kota"}).copy()

# Normalisasi: 'Bali' → 'Denpasar' agar match (konservatif)
wiki_join["kota"] = wiki_join["kota"].replace({"Bali":"Denpasar"})

df_merge = pd.merge(
    wiki_join, api_agg,
    on="kota", how="inner", validate="many_to_one",
    suffixes=("_masakan","_cuaca")
)

print(f"Hasil merge  : {df_merge.shape[0]} baris x {df_merge.shape[1]} kolom")
n_dup = df_merge.duplicated().sum()
print(f"Baris duplikat penuh: {n_dup}")
assert n_dup == 0, "Duplikasi baris ditemukan!"

FILE_MERGE = "data/processed/merge_kuliner_bali.csv"
df_merge.to_csv(FILE_MERGE, index=False, encoding="utf-8")

# Metadata hasil
metadata = {
    "judul"     : "Kuliner Bali × Cuaca",
    "sumber"    : [p["sumber"] for p in PROVENANCE],
    "baris"     : int(df_merge.shape[0]),
    "kolom"     : list(df_merge.columns),
    "md5_hasil" : md5_file(FILE_MERGE),
    "waktu"     : datetime.now(timezone.utc).isoformat(),
}
Path("data/processed/merge_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")

Path("docs/provenance.json").write_text(
    json.dumps(PROVENANCE, indent=2, ensure_ascii=False), encoding="utf-8")

# === DATA ETHICS CHECKLIST 9 BUTIR ===
ethics = pd.DataFrame([
    ("1. Lisensi jelas",            "Lengkap", "BPS: CC BY 4.0; Open-Meteo: CC BY 4.0; Wikipedia: CC BY-SA 4.0"),
    ("2. Sumber terverifikasi",     "Lengkap", "3 URL domain resmi"),
    ("3. Robots.txt diperiksa",     "Lengkap", f"id.wikipedia.org izin: {izin}"),
    ("4. PII langsung",             "Aman",    "Tidak ada; data agregat & publik"),
    ("5. Quasi-identifier",         "Tercatat","Kolom provinsi/kota/tahun = QI"),
    ("6. User-Agent kontak",        "Lengkap", USER_AGENT),
    ("7. Timeout & delay",          "Lengkap", f"timeout={TIMEOUT}s, delay={DELAY}s"),
    ("8. Retry & error handling",   "Lengkap", f"max_retry={MAX_RETRY}, backoff linear"),
    ("9. Provenance & MD5",         "Lengkap", "docs/provenance.json"),
], columns=["Butir","Status","Bukti"])
ethics.to_csv("docs/data_ethics_checklist.csv", index=False)
print("\n=== DATA ETHICS CHECKLIST ===")
print(ethics.to_string(index=False))

print(f"\n[OK] Merge      : {FILE_MERGE}")
print(f"[OK] Metadata   : data/processed/merge_metadata.json")
print(f"[OK] Provenance : docs/provenance.json")
print(f"[OK] Ethics     : docs/data_ethics_checklist.csv")

=== MERGE & VALIDASI ===
Hasil merge  : 75 baris x 7 kolom
Baris duplikat penuh: 0

=== DATA ETHICS CHECKLIST ===
                    Butir   Status                                                          Bukti
         1. Lisensi jelas  Lengkap BPS: CC BY 4.0; Open-Meteo: CC BY 4.0; Wikipedia: CC BY-SA 4.0
  2. Sumber terverifikasi  Lengkap                                             3 URL domain resmi
  3. Robots.txt diperiksa  Lengkap                                    id.wikipedia.org izin: True
          4. PII langsung     Aman                               Tidak ada; data agregat & publik
      5. Quasi-identifier Tercatat                                 Kolom provinsi/kota/tahun = QI
     6. User-Agent kontak  Lengkap             UTS-DAM-INSTIKI/2026 (kontak: gungyoga1@gmail.com)
       7. Timeout & delay  Lengkap                                        timeout=15s, delay=2.0s
8. Retry & error handling  Lengkap                                    max_retry=3, backoff linear
    

In [8]:
# =====================================================
# SEL 7 — Ringkasan akhir + estimasi total storage
# =====================================================
semua_berkas = [FILE_BPS, FILE_API_CSV, FILE_API_JSON, FILE_WIKI, FILE_MERGE]
total_bytes = sum(Path(p).stat().st_size for p in semua_berkas if Path(p).exists())
total_mb = total_bytes / (1024**2)
per_tahun = total_mb * 12
cadangan = per_tahun * 0.30
print("=== ESTIMASI STORAGE TOTAL 1 TAHUN ===")
print(f"Ukuran semua sumber : {total_mb:.4f} MB")
print(f"x 12 bulan          : {per_tahun:.4f} MB")
print(f"+ cadangan 30%      : {cadangan:.4f} MB")
print(f"TOTAL/tahun         : {per_tahun + cadangan:.4f} MB (≈ {(per_tahun+cadangan)/1024:.6f} GB)")
print("\nAsumsi: snapshot bulanan; cadangan untuk backup + olahan + growth; tanpa indeks DB.")

print("\n=== SELESAI ===")
print("Semua soal (1–5) telah dikerjakan. Cek folder data/ dan docs/ untuk output.")

=== ESTIMASI STORAGE TOTAL 1 TAHUN ===
Ukuran semua sumber : 0.3380 MB
x 12 bulan          : 4.0554 MB
+ cadangan 30%      : 1.2166 MB
TOTAL/tahun         : 5.2721 MB (≈ 0.005149 GB)

Asumsi: snapshot bulanan; cadangan untuk backup + olahan + growth; tanpa indeks DB.

=== SELESAI ===
Semua soal (1–5) telah dikerjakan. Cek folder data/ dan docs/ untuk output.


In [10]:
# =====================================================
# SEL 8 — Bungkus semua hasil ke ZIP & unduh otomatis
# =====================================================
import shutil
import zipfile
from pathlib import Path

with open("README.md", "w", encoding="utf-8") as f:
    f.write("""# UTS Data Acquisition & Management — Kuliner Tradisional Bali

**Nama:** Gung Yoga
**Email:** gungyoga1@gmail.com
**Topik:** Kuliner Tradisional Bali
**Tanggal:** Kamis, 19 November 2026

## Sumber data
1. **BPS** — CSV usaha makanan/minuman (CC BY 4.0)
2. **Open-Meteo API** — cuaca 3 kota Bali (CC BY 4.0)
3. **Wikipedia ID** — tabel masakan Bali (CC BY-SA 4.0, dicek robots.txt)

## Output
- `data/raw/` — data mentah (tidak diubah)
- `data/processed/` — hasil merge + metadata
- `docs/` — provenance, ethics checklist, robots.txt

## Cara jalankan
Buka `UTS_DAM_kuliner_bali.ipynb` di Colab/Jupyter → Run all.
""")

shutil.make_archive("UTS_DAM_KulinerBali_gungyoga1", "zip", ".",
                    "data")  # zip hanya folder data
# Tambahkan docs & README ke zip yang sama menggunakan zipfile.ZipFile
with zipfile.ZipFile("UTS_DAM_KulinerBali_gungyoga1.zip", "a") as z:
    z.write("README.md")
    for p in Path("docs").rglob("*"):
        if p.is_file():
            z.write(p)

from google.colab import files
files.download("UTS_DAM_KulinerBali_gungyoga1.zip")
print("[OK] ZIP siap diunduh.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[OK] ZIP siap diunduh.
